In [0]:
dbutils.library.restartPython()

In [0]:

from common_utils.logging import get_logger
from common_utils.ingester import read_sql_jdbc , write_raw
from datetime import date
import json 

# =========================================
# 0.Logger
# =========================================
logger = get_logger('func_sqlserver.ipynb')
logger.info('Session started')

# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 
print(config_path)
# =========================================
# 2. Read Config
# =========================================
with open(config_path,"r") as f:
    config = json.load(f)


# =========================================
# 3. CONFIG Sections
# =========================================
source_config = config['source']
target_config = config['target']
write_config = config['write_options']

# =========================================
# 4. Run Date
# =========================================

run_date = date.today().isoformat()

# =========================================
# 5. Getting passwords from secret scope
# =========================================
user=dbutils.secrets.get(scope="retail-platform-dev",key=source_config["user"])
password=dbutils.secrets.get(scope="retail-platform-dev",key=source_config["password"])
# =========================================
# 6. READ FORM SQL
# =========================================

df = read_sql_jdbc(spark,source_config['url'],source_config["dbtable"], user, password)
#get_logger('rows %s in the Dataframe', df.count())
logger.info('sample Data')
df.display()
target_path = f'{target_config['base_path']}/{target_config['folder']}/load_date={run_date}'
logger.info(f'writing the datafram to {target_path}')
logger.info('load date is %s', run_date)
write_raw(df,target_path, target_config['mode'],target_config['file_format'], write_config)
logger.info('Data landed for %s at %s',run_date,target_path)
logger.info('session completed')
